# ДЗ 8. Physics-Informed Neural Networks

На лекции вы уже реализовали (`SimplestPINN.ipynb`) прямую задачу для уравнения теплопроводности
$u_t = u_{xx}$ и (`InversePINN.ipynb`) обратную задачу восстановления вязкости $\nu$ в уравнении
Бюргерса. В `pres_2.tex` там же было дано самостоятельное задание — уравнение Пуассона,
теплопроводность с источником, восстановление $\nu$ в $u_t=\nu u_{xx}$ по зашумлённым данным.
**Эти три задачи здесь не повторяются** — предполагается, что вы их уже решили или решаете
параллельно с этим ДЗ.

Вместо этого мы превращаем **вопросы для размышления** из той же лекции в реальные эксперименты:
почему PINN не достигают машинной точности? Как влияет мало точек на границе? Как влияет замена
`tanh` на `ReLU`? В каком случае вы выберете классический численный метод, а не PINN? У каждого из
этих вопросов в лекции не было ответа — здесь мы отвечаем на них экспериментально.

Все задачи используют одномерные по пространству уравнения (плюс время) — обучение занимает от
десятков секунд до нескольких минут на CPU.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
torch.manual_seed(0)

class PINN(nn.Module):
    def __init__(self, layers, activation=nn.Tanh):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(layers[i], layers[i+1]) for i in range(len(layers)-1)])
        self.act = activation()
    def forward(self, x, t):
        out = torch.cat([x, t], dim=1)
        for i, layer in enumerate(self.layers):
            out = layer(out)
            if i < len(self.layers) - 1:
                out = self.act(out)
        return out

def gradients(outputs, inputs):
    return torch.autograd.grad(outputs, inputs, grad_outputs=torch.ones_like(outputs),
                                create_graph=True, retain_graph=True)[0]

## Уровень 1. Попроще

### Задача 1.1. Уравнение переноса (адвекции)

Уравнение переноса $u_t + c\,u_x = 0$, $u(x,0)=\sin(\pi x)$, $x \in [-1,1]$ с **периодическими**
граничными условиями ($u(-1,t)=u(1,t)$ — волна, уходящая за правый край, "появляется" слева).
Точное решение — бегущая волна: $u(x,t) = \sin(\pi(x - ct))$ (с периодическим приведением
$x-ct$ к $[-1,1]$).

1. Реализуйте PINN по образцу `SimplestPINN.ipynb`, но с новым PDE-residual
   ($u_t + c u_x$ вместо $u_t - u_{xx}$ — обратите внимание, уравнение первого порядка по $x$,
   вторая производная не нужна) и **периодическим** граничным условием вместо условия Дирихле
   (лосс за несовпадение $u(-1,t)$ и $u(1,t)$, а не за отклонение от нуля).
2. Сравните с точным решением в нескольких точках по времени.
3. Постройте график $u(x,t)$ как функцию $x$ при нескольких $t$ — убедитесь, что волна визуально
   "едет" в правильном направлении с правильной скоростью $c$.

In [ ]:
# TODO: PINN для уравнения переноса с периодическим BC

### Задача 1.2. Активация: `tanh` против `ReLU`

В вопросах для размышления на лекции спрашивалось: "Как повлияет замена активации `tanh` на
`ReLU` на гладкость решения?" — проверьте это экспериментально.

1. Обучите PINN для уравнения теплопроводности (как в `SimplestPINN.ipynb`) дважды: с `nn.Tanh` и
   с `nn.ReLU`, при одинаковой архитектуре и числе эпох.
2. Сравните итоговую ошибку **и** визуальную гладкость производной $\partial u/\partial x$,
   посчитанной через `autograd`.
3. Объясните результат, опираясь на то, что PDE-residual включает **вторую** производную $u_{xx}$.

In [ ]:
# TODO: сравнение tanh vs ReLU для теплопроводности

### Задача 1.3. Балансировка слагаемых лосса

Суммарный лосс PINN складывается из разнородных слагаемых, которые могут иметь очень разный
масштаб. Если одно из них систематически меньше остальных, градиентный спуск фактически его
игнорирует (тот же принцип несбалансированных величин, что и в ДЗ 2/6).

1. Обучите PINN для теплопроводности с **намеренно заниженным** весом BC:
   `loss = loss_pde + 0.001*loss_bc + loss_ic`.
2. Сравните итоговое нарушение граничного условия ($|u(\pm1,t)|$, в норме должно быть $\approx0$)
   с версией с равными весами.
3. Попробуйте обратную крайность — сильно завышенный вес BC/IC. Что происходит с PDE-residual?

In [ ]:
# TODO: обучение с разными весами слагаемых лосса, сравнение нарушения BC

## Уровень 2. Средний

### Задача 2.1. PINN против конечных разностей: когда какой метод выбрать

Лекция прямо спрашивала: "В каком случае вы выберете FEM, а в каком — PINN?" — сравните PINN с
классическим методом конечных разностей (finite differences, FD) на **одной и той же** задаче
(уравнение теплопроводности из `SimplestPINN.ipynb`), где у FD нет никаких принципиальных
трудностей — это должно стать наглядной demonstраcией "в каком случае вы выберете FD".

1. Реализуйте явную конечно-разностную схему для $u_t=u_{xx}$: `u_new[i] = u[i] + dt/dx^2*(u[i+1]
   - 2u[i] + u[i-1])`, с шагом по времени, удовлетворяющим условию устойчивости
   $dt \le 0.5\,dx^2$ (иначе явная схема разойдётся — если интересно, попробуйте нарушить условие
   и посмотрите, что будет).
2. Сравните с PINN (или уже обученной моделью из задачи 1.2) по: (а) точности относительно
   точного решения, (б) времени вычисления.
3. Сформулируйте (текстом) 2-3 сценария, где, наоборот, PINN был бы предпочтительнее FD
   (подсказки: обратные задачи? данные вместо/вместе с уравнением? нерегулярная геометрия? высокая
   размерность?).

In [ ]:
# TODO: конечно-разностная схема + сравнение с PINN по точности и времени

### Задача 2.2. Плотность точек коллокации на границе

Лекция спрашивала: "Что произойдёт, если взять слишком мало точек для граничных условий?"
Проверьте явно, систематически меняя `N_b`.

1. Обучите PINN для теплопроводности с $N_b \in \{2, 10, 50, 200\}$ точками на границе (при
   фиксированном $N_f$ внутри области), одинаковое число эпох для всех.
2. Постройте график: нарушение граничного условия ($\max|u(\pm1,t)|$ на плотной тестовой сетке
   по $t$) от $N_b$.
3. При $N_b=2$ (минимально возможное — по одной точке на каждую границу) видно ли переобучение
   именно **в этих конкретных точках** при плохом обобщении на остальные значения $t$ вдоль
   границы?

In [ ]:
# TODO: обучение с разным N_b, график нарушения BC от N_b

### Задача 2.3. Прямая задача Бюргерса: острый фронт — известная трудность для PINN

`InversePINN.ipynb` решал **обратную** задачу Бюргерса. Здесь — **прямая** задача с той же
физикой: $u_t + u\,u_x = \nu\,u_{xx}$, $u(x,0) = -\sin(\pi x)$, $u(\pm1,t)=0$, классический
бенчмарк Raissi et al. (2019). При малой вязкости $\nu = 0.01/\pi$ решение развивает **острый
фронт** (почти разрыв) около $x=0$ при $t \to 1$ — известная трудность именно для PINN (обычная
конечно-разностная схема с адаптивной сеткой справляется с этим существенно проще).

1. Реализуйте PINN для этого уравнения (структура кода очень похожа на `InversePINN.ipynb`, но
   $\nu$ теперь **известна и фиксирована**, не выучивается).
2. Обучите при **умеренном** числе эпох (например, 5000-8000, чтобы уложиться в бюджет) и
   визуализируйте решение как функцию $x$ при нескольких моментах времени, включая $t$ близкое к 1.
3. Посмотрите на PDE-residual **как функцию $x$** при фиксированном $t\approx1$ (не усреднённое
   число, а именно график) — увидите ли вы всплеск residual именно в районе острого фронта
   ($x\approx0$)?

In [ ]:
# TODO: прямая задача Бюргерса, визуализация фронта и residual

## Уровень 3. Со звёздочкой

### Задача 3.1 (★★). Обратная по времени теплопроводность: некорректно поставленная задача

Не всякая обратная задача одинаково сложна. Обратная по времени диффузия (знаем состояние в
некоторый **более поздний** момент, хотим восстановить более раннее/начальное) — классический
пример **некорректно поставленной** (ill-posed) задачи: сколь угодно малое возмущение данных
может привести к сколь угодно большой ошибке в решении.

**Часть А — аналитическая демонстрация (без всякого обучения, за секунды).** Representация
через ряд Фурье: $u(x,t) = \sum_k a_k e^{-k^2\pi^2 t}\sin(k\pi x)$. Если по данным при $t=T$
восстанавливать коэффициенты при $t=0$ напрямую (разделить на $e^{-k^2\pi^2 T}$, то есть
**умножить** на $e^{+k^2\pi^2 T}$), посчитайте, во сколько раз усиливается ошибка каждой моды
$k$, и постройте график этого коэффициента усиления от $k$ при нескольких $T$.

**Часть Б — попытка через PINN.** Обучите PINN, аналогичный `SimplestPINN.ipynb`, но с данными,
заданными **не** при $t=0$ (как обычно), а при $t=T_{final}$ (с небольшим шумом), плюс то же
самое PDE-уравнение и граничные условия при $t\in[0,T_{final}]$. Попробуйте восстановить
$u(x,0)$. Сравните качество восстановления с обычной прямой задачей (данные при $t=0$, требуется
найти $u(x,T_{final})$) при том же уровне шума.

In [ ]:
# Часть А: коэффициент усиления ошибки для разных гармоник и времён
T_values = [0.05, 0.1, 0.3]
ks = np.arange(1, 15)

plt.figure(figsize=(6, 4))
for T in T_values:
    amplification = np.exp(ks**2 * np.pi**2 * T)
    plt.semilogy(ks, amplification, 'o-', label=f'T={T}')
plt.xlabel('номер гармоники k'); plt.ylabel('коэффициент усиления ошибки (лог. шкала)')
plt.legend(); plt.title('Обратная теплопроводность: усиление ошибки по модам')
plt.show()

for k in [1, 5, 10]:
    print(f"k={k}, T=0.3: усиление шума в {np.exp(k**2*np.pi**2*0.3):.3e} раз")

In [ ]:
# TODO часть Б: PINN для обратной по времени теплопроводности

### Задача 3.2 (★★). Жёсткие ограничения (hard constraints) вместо штрафов

Вместо того чтобы **штрафовать** сеть за нарушение BC/IC через дополнительные слагаемые лосса
(и тратить силы на балансировку весов, как в задаче 1.3), можно построить архитектуру, которая
удовлетворяет BC/IC **точно, по построению**, вне зависимости от весов сети — это убирает саму
необходимость в компромиссе.

Для нашей задачи ($u(x,0)=\sin(\pi x)$, $u(\pm1,t)=0$): постройте выход сети как
$$u(x,t) = \sin(\pi x) + t\cdot(1-x^2)\cdot N\!N(x,t)$$
где $NN$ — обычная полносвязная сеть. Проверьте: при $t=0$ второе слагаемое обнуляется (множитель
$t$), и $u(x,0)=\sin(\pi x)$ **точно**. При $x=\pm1$ множитель $(1-x^2)=0$, и $u(\pm1,t)=
\sin(\pm\pi)=0$ **точно** — оба условия выполнены аналитически, для любых весов $NN$.

1. Реализуйте эту архитектуру и обучайте её, минимизируя **только** PDE-residual (без слагаемых
   `loss_bc`/`loss_ic` — они больше не нужны!).
2. Убедитесь программно (не только "по построению"), что BC и IC действительно выполняются с
   точностью до чисел с плавающей точкой на множестве тестовых точек.
3. Сравните итоговую точность решения (относительно точного $u(x,t)=e^{-\pi^2t}\sin(\pi x)$) с
   обычной "мягкой" версией (задача 1.2 или 1.3) при **одинаковом** числе эпох.

In [ ]:
# TODO: PINNHardConstraint, обучение только по PDE-residual, проверка BC/IC

### Задача 3.3 (★). L-BFGS дообучение: приближение к пределу точности

Классический рецепт из литературы по PINN (в том числе оригинальной статьи Raissi et al.):
сначала обучать Adam-ом (быстрая, устойчивая, но не самая точная сходимость), затем **дообучить**
методом **L-BFGS** — квазиньютоновским методом (вспомните ДЗ 1, задачи про метод Ньютона и
Hessian-free оптимизацию!) — который на финальном этапе, когда мы уже близко к минимуму, часто
даёт заметно более точную сходимость, чем продолжение Adam.

1. Обучите PINN для теплопроводности Adam-ом заданное число эпох (как обычно), запомните
   итоговую ошибку.
2. Дообучите **тот же** экземпляр модели с помощью `torch.optim.LBFGS` (нужен `closure` —
   функция, пересчитывающая loss и делающая `backward()`, так как L-BFGS может вызывать её
   несколько раз за один шаг).
3. Сравните точность до и после L-BFGS, а также время, затраченное на L-BFGS-фазу относительно
   Adam-фазы.

In [ ]:
# TODO: Adam, затем L-BFGS дообучение, сравнение точности

---
**Что сдавать:** ноутбук с реализованными PINN, графиками и письменными ответами на вопросы для
решённых задач. Каждая задача по отдельности обучается от десятков секунд до пары минут на CPU;
суммарно все 9 задач — порядка 15-25 минут. Часть А задачи 3.1 не требует обучения вообще
(чистый анализ Фурье-мод) и считается мгновенно.